# 03 Test

This notebook runs end-to-end prompt tests against the APIM gateway endpoint.

## Load Environment

In [1]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path('../env/.env')
if not env_path.exists():
    raise RuntimeError('env/.env not found. Run 01_deploy_infra.ipynb first.')

load_dotenv(env_path)
APIM_ENDPOINT = os.getenv('APIM_ENDPOINT', '').rstrip('/')
APIM_PATH = os.getenv('APIM_CHAT_COMPLETIONS_PATH', '/openai/chat/completions')
APIM_API_KEY = os.getenv('APIM_API_KEY', '')

print(f'APIM endpoint: {APIM_ENDPOINT}')
print(f'APIM path: {APIM_PATH}')

APIM endpoint: https://apim-evalgw02-1-uks.azure-api.net
APIM path: /openai/chat/completions


## Send Prompt Set Through APIM

In [3]:
import requests

if not APIM_API_KEY or APIM_API_KEY.startswith('[Retrieve from Azure Portal'):
    raise RuntimeError('APIM_API_KEY is not set with a real key in env/.env')

test_prompts = [
    'What is the capital of France?',
    'Explain machine learning in one sentence.',
    'Name three primary colors.'
]

results = []
url = f"{APIM_ENDPOINT}{APIM_PATH}"
headers = {
    'Ocp-Apim-Subscription-Key': APIM_API_KEY,
    'Content-Type': 'application/json'
}

for i, prompt in enumerate(test_prompts, start=1):
    payload = {
        'messages': [
            {'role': 'system', 'content': 'You are a concise assistant.'},
            {'role': 'user', 'content': prompt}
        ],
        'max_tokens': 120
    }
    response = requests.post(url, headers=headers, json=payload, timeout=45)

    response_model = ''
    if response.status_code == 200:
        body = response.json()
        answer = body.get('choices', [{}])[0].get('message', {}).get('content', '')
        response_model = body.get('model', '')
        status = 'success'
    else:
        answer = response.text[:600]
        status = f'error_{response.status_code}'

    row = {
        'index': i,
        'query': prompt,
        'response': answer,
        'model': response_model,
        'status': status
    }
    results.append(row)

    print(f'\n--- Test {i} ---')
    print(f'Status: {status}')
    print(f'Model: {response_model if response_model else "[not returned]"}')
    print(f'Input: {prompt}')
    print('Output:')
    print(answer.strip() if answer else '[empty response]')
    print('---------------')

results_path = Path('../outputs/test-results.json')
results_path.parent.mkdir(parents=True, exist_ok=True)
results_path.write_text(json.dumps(results, indent=2), encoding='utf-8')

success_count = sum(1 for r in results if r['status'] == 'success')
print(f'\nCompleted tests: {success_count}/{len(results)} succeeded')
print(f'Results file: {results_path}')


--- Test 1 ---
Status: success
Model: gpt-4o-2024-11-20
Input: What is the capital of France?
Output:
The capital of France is Paris.
---------------

--- Test 2 ---
Status: success
Model: gpt-4o-2024-11-20
Input: Explain machine learning in one sentence.
Output:
Machine learning is a subset of artificial intelligence where computers learn patterns and make predictions or decisions from data without explicit programming.
---------------

--- Test 3 ---
Status: success
Model: gpt-4o-2024-11-20
Input: Name three primary colors.
Output:
The three primary colors are **red**, **blue**, and **yellow**.
---------------

Completed tests: 3/3 succeeded
Results file: ..\outputs\test-results.json
